In [ ]:
import torch
import math
import uuid
from pathlib import Path
from datetime import datetime 
import json
import time
import random

In [ ]:
# PLAN: First we create a plan which tells Arcana what, when, where meansurements should be recorded and aggregated

plan_v1 = {
    'model':{
        'gradients':{
            'frequency': 'per_batch',   # per_epoch, per_n_epoch, per_n_batch
            'spatial_granularity': 'representative',  # representative, all, selected
            'temporal_granularity': 'window',   # window, None
            'stats': ['mean', 'l2_norm', 'rms_norm'],
            'seed': 42,
            'num_layers': 5
        },
        'activations':{
            'frequency': 'per_batch',   # per_epoch, per_n_epoch, per_n_batch
            'spatial_granularity': 'representative',  # representative, all, selected
            'temporal_granularity': 'window',   # window, None
            'stats': ['mean', 'l2_norm', 'rms_norm'], 
            'seed': 42,
            'num_layers': 5
        }
    },
    'system':{

    },
    'window':{
        'size_in_steps':10
    }
}

In [ ]:
# This class is a wrapper around dataloader that tracks batch number without needing to pass context managers to training and evaluation scripts 

class TrackLoader:
    def __init__(self, run, dataloader, epoch, phase='train'):
        self.phase = phase
        self.epoch = epoch
        self.dataloader = dataloader
        self.run = run

    def __len__(self):
        return len(self.dataloader)

    def __iter__(self):
        self.run.begin_phase(self.epoch, self.phase)
        try:
            for batch_idx, data in enumerate(self.dataloader):
                self.run.begin_batch(batch_idx)
                yield data
        finally:
            self.run.end_phase()

In [ ]:
# BUILDING THE CONTEXT MANAGER

class Arcana():
    def __init__(self, plan):
        Path('log-run').mkdir(parents=False, exist_ok=True)
        self.plan = plan

    def run(self, mode, dir='log-run'):
        return Run(mode=mode, dir=dir, plan=self.plan)

class Run():
    def __init__(self, mode, dir, plan):
        self.id = uuid.uuid4()
        self.timestamp = datetime.now()
        self.mode=mode
        self.dir=Path(dir)
        self.status='created'
        self.plan=plan
        self.phase=None
        self.batch=0
        self.epoch=0
        self.accum_epoch={'activation':{}, 'gradient':{}}
        self.global_step=-1
        self.window_range=[None, None]
        self.tracked_modules=[]
        self.handlers = []
        self.run_record={
            'id': str(self.id),
            'timestamp': str(self.timestamp),
            'mode': mode,
            'status':self.status,
            'activation':[],
            'attention':[],
            'gradient':[],
            'train_loss':[],
            'train_acc':[],
            'eval_loss':[],
            'eval_acc':[],
            'perplexity':[]
        }

    # THIS FUNCTION EXECUTES WHEN CONTROL ENTERS CONTEXT MANAGER
    def __enter__(self):
        if self.run_record['status'] != 'created':
            raise RuntimeError('Run already started')
        
        self.status='running'
        self.run_record['status']=self.status
        self.start_run=time.perf_counter()
        print(f'Run has started arcana will now record. Run id is {self.id}')
        return self

    # THIS FUNCTION EXECUTES WHEN CONTROL EXITS CONTEXT MANAGER 
    def __exit__(self, exc_type, exc, tb):
        for handler in self.handlers:
            handler.remove()

        self.handlers.clear()

        self.run_record['run_duration']=time.perf_counter()-self.start_run

        self.run_record['status']='failed' if exc_type else 'completed'

        filename = f'{self.id}--{datetime.now().strftime("%Y-%m-%d_%H-%M-%S-%f")}.json'
        with open(Path(self.dir/filename), 'w', encoding='utf-8') as f:
            json.dump(self.run_record, f, indent=4)

        print(f"Arcana: Run {self.id} finished")

    # THE FUNCTION THAT HANDLES ALL THE GRADIENT AND ACTIVATION TRACKING SHITTT
    def watch(self, model):
        if self.run_record['status'] != 'running':
            raise RuntimeError('Run must be active before starting watch')
        activation_config=self.plan['model']['activations']
        gradient_config=self.plan['model']['gradients']

        shared_modules = None

        if activation_config.get('spatial_granularity') == 'representative' and gradient_config.get('spatial_granularity') == 'representative':
            shared_modules = self.__select_modules(model, activation_config)
        activ_modules = self.__select_modules(model, activation_config, shared_cache=shared_modules)
        grad_modules = self.__select_modules(model, gradient_config, shared_cache=shared_modules)

        self.run_record['tracked_modules']={
            'activation': [name for name, _ in activ_modules],
            'gradient': [name for name, _ in grad_modules]
        }

        for name, module in activ_modules:
            activ_handler = module.register_forward_hook(self.make_hook(name, 'activation', activation_config))
            self.handlers.append(activ_handler)

        for name, module in grad_modules:
            grad_modules = module.register_full_backward_hook(self.make_hook(name, 'gradient', gradient_config))
            self.handlers.append(grad_modules)              

    # THESE ARE HELPER FUNCTIONS
    def track_loader(self, dataloader, epoch, phase):
        return TrackLoader(self, dataloader, epoch, phase)

    def begin_phase(self, epoch, phase):
            self.phase=phase
            self.epoch=epoch
            if self.phase == 'train':
                self.window_range=[None, None]
                self.accum_epoch={'activation':{}, 'gradient':{}}

    def end_phase(self):
        if self.phase == "train":
            self.flush_epoch_stats('activation', self.plan['model']['activations'])
            self.flush_epoch_stats('gradient', self.plan['model']['gradients'])
        self.phase = None

    def begin_batch(self, batch):
        self.batch=batch
        if self.phase == 'train':
            self.global_step+=1
            win=self.current_window()
            if self.window_range[0] is None:
                self.window_range[0]=win
            self.window_range[1]=win

    def is_window_enabled(self,cfg):
        win_cfg=self.plan.get('window')
        return(win_cfg is not None and win_cfg.get('size') is not None and cfg.get('temporal_aggregation') == 'window')

    def current_window(self):
        win_cfg=self.plan.get('window')
        if not win_cfg and win_cfg.get('size'):
            return
        return self.global_step // win_cfg.get('size')

    @staticmethod
    def extract_tensor(t):
        if isinstance(t, torch.tensor):
            return t
        if isinstance(t, (tuple, list)):
            for item in t:
                if isinstance(item, torch.tensor):
                    return item
        return None


    @staticmethod
    def __select_modules(model, cfg, shared_cache):
        scope=cfg.get('spatial_granularity')
        leaves=[(name, mod) for name, mod in model.named_modules() if len(list(mod.childern())) == 0]
        if scope == 'all':
            return leaves
        elif scope == 'representative':
            if shared_cache is None:
                m = min(cfg.get('num_layers', 5), len(leaves))
                rand_gen = random.Random(cfg.get('seed', 42))
                rand_idx = sorted(rand_gen.sample(len(leaves), m))
                return [leaves[i] for i in rand_idx]
            return shared_cache
        elif scope == 'selected':
            list=model.named_parameters()
        else:
            raise RuntimeError(f'Invalid value {scope}. Choose from {['all', 'representative', 'selected']}')


    def make_hook(self, name, record_key, cfg):
        frequency=cfg.get('frequency')
        def hook(module, input, output):
            if self.phase != 'train':
                return
            tensor = self.extract_tensor(output)
            if tensor.numel() == 0 or tensor is None:
                raise RuntimeError('Tensor is empty')
            t = tensor.detach()
            if frequency == 'per_batch':
                entry = {
                    'epoch': self.epoch,
                    'batch': self.batch,
                    'global_step': self.global_step,
                    'timestamp': str(datetime.now()),
                    'module': {
                        'module_name': name,
                        'stats': self.__stats(t, cfg['stats'])
                    }
                }
                if self.is_window_enabled(cfg):
                    entry['window'] = self.current_window()
                self.run_record[record_key].append(entry)
            elif frequency == 'per_epoch':
                self.accum_epoch_stats()
            else:
                raise ValueError(f'Unsupported frequency type: {frequency}. Choose from {['per_epoch', 'per_batch']}')
        return hook


    def __stats(self, value, stats):
            t=value.float()
            allowed_stats = ['mean', 'l2', 'rms']
            for stat in stats:
                if stat in allowed_stats:
                    if stat == 'mean':
                        mean=t.mean().item()
                    if stat == 'l2':
                        l2=torch.sqrt(torch.sum(t**2)).item()
                    if stat == 'rms':
                        rms = torch.sqrt(torch.mean(t**2)).item()
                else:
                    raise RuntimeError(f'Invalid Stat option provided. Available Options: {allowed_stats}')
            return {
                'shape':list(t.shape),
                'mean':mean,
                'l2-norm': l2,
                'rms': rms,
            }

    def accum_epoch_stats(self, name, record_key, t):
        acc = self.accum_epoch[record_key].setdefault({
            name, {'shape':t.shape, 'numel':0, 'sum':0.0, 'l2_sum':0.0, 'sq_sum':0.0, 'batches':0}
        })
        norm = torch.linalg.vector_norm(t).item()
        acc['numel'] += t.numel()
        acc['sum'] += t.sum()
        acc['l2_sum'] += norm
        acc['sq_sum'] += (t**2).sum()
        acc['batches'] += 1

    def flush_epoch_stats(self, cfg, record_key):
        if cfg['frequency'] != 'per_epoch':
            return
        for name, agg in self.accum_epoch[record_key].items():
            stats = {'shape':agg['shape']}
            for stat in cfg['stat']:
                if stat == 'mean':
                    stats['mean'] = agg['sum']/agg['numel']
                if stat == 'l2':
                    stats['l2-norm'] = agg['l2_sum']/agg['batches']
                if stat == 'rms':
                    stats['rms'] = math.sqrt(agg['l2_sum']/agg['numel'])
                entry = {
                    'epoch': self.epoch,
                    'batches_aggregated': agg['batches'],
                    'timestamp': str(datetime.now()),
                    'module': {
                        'module_name': name,
                        'stats': stats
                    }
                }
                self.run_record[record_key].append(entry)

    def log_metrics(self):
        pass

    def log_system(self):
        pass
    